# KNN tuning

KNN gets a compact, distance-friendly input contract instead of the larger Logistic Regression
contract. Categories are one-hot encoded, all resulting dimensions are scaled, and the full
grid is evaluated with collision-group-aware folds. KNN in scikit-learn is CPU based; a GPU is
not used by this notebook. The best pipeline includes its encoder and scaler, so Streamlit can
call it with raw columns.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, recall_score, make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists():
    ROOT = ROOT.parent
TRAIN_PATH = ROOT / 'dataset' / 'processed' / 'merged_train.csv'
TEST_PATH = ROOT / 'dataset' / 'processed' / 'merged_test.csv'
RESULTS = ROOT / 'hyper parameter' / 'results' / 'knn'
BEST = ROOT / 'hyper parameter' / 'best'
MODEL_DIR = ROOT / 'models' / 'saved_models'
RESULTS.mkdir(parents=True, exist_ok=True)
BEST.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 19
TARGET = 'casualty_severity'
GROUP = 'collision_index'
MODEL_NAME = 'knn'

In [ ]:
FEATURES = [
    'casualty_class', 'sex_of_casualty', 'age_of_casualty', 'casualty_type',
    'number_of_vehicles', 'first_road_class', 'road_type', 'speed_limit',
    'light_conditions', 'weather_conditions', 'urban_or_rural_area',
    'vehicle_type', 'vehicle_manoeuvre', 'vehicle_leaving_carriageway',
    'first_point_of_impact', 'skidding_and_overturning',
    'hit_object_in_carriageway', 'hit_object_off_carriageway',
    'age_of_driver', 'engine_capacity_cc',
]
NUMERIC = ['age_of_casualty', 'age_of_driver', 'engine_capacity_cc', 'number_of_vehicles', 'speed_limit']
CATEGORICAL = [column for column in FEATURES if column not in NUMERIC]
train = pd.read_csv(TRAIN_PATH, dtype={GROUP: str})
test = pd.read_csv(TEST_PATH, dtype={GROUP: str})
missing = sorted(set(FEATURES) - set(train.columns))
if missing:
    raise ValueError(f'Missing prepared columns: {missing}. Run build_dataset.py first.')
X = train[FEATURES]
y = train[TARGET].astype(int)
groups = train[GROUP].astype(str)
X_test = test[FEATURES]
y_test = test[TARGET].astype(int)
print('train:', X.shape, 'test:', X_test.shape)
print('class proportions:', y.value_counts(normalize=True).sort_index().round(6).to_dict())

In [ ]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', min_frequency=10), CATEGORICAL),
    ('numeric', 'passthrough', NUMERIC),
], remainder='drop', sparse_threshold=1.0)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scale', StandardScaler(with_mean=False)),
    ('classifier', KNeighborsClassifier(n_jobs=-1)),
])
# Compact one-hot space avoids distance dilution from weak columns.
PARAM_GRID = {
    'classifier__n_neighbors': [3, 5, 9, 15, 25, 41, 61],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2],
    'classifier__algorithm': ['auto'],
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = list(cv.split(X, y, groups))
for fold, (_, validation_index) in enumerate(cv_splits):
    fold_rate = y.iloc[validation_index].eq(1).mean()
    if abs(fold_rate - y.eq(1).mean()) > 0.001:
        raise ValueError(f'Fatal proportion is outside tolerance in fold {fold}')
print('folds:', len(cv_splits), 'parameter combinations:', 7 * 2 * 2)

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'macro_f1': 'f1_macro',
    'fatal_recall': make_scorer(recall_score, labels=[1], average='macro', zero_division=0),
    'serious_recall': make_scorer(recall_score, labels=[2], average='macro', zero_division=0),
    'slight_recall': make_scorer(recall_score, labels=[3], average='macro', zero_division=0),
}
search = GridSearchCV(
    estimator=pipeline,
    param_grid=PARAM_GRID,
    scoring=scoring,
    refit='macro_f1',
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score='raise',
)
search.fit(X, y, groups=groups)
cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_macro_f1')
cv_results.to_csv(RESULTS / 'cv_results.csv', index=False)
print('best parameters:', search.best_params_)
print('best grouped macro-F1:', search.best_score_)

In [ ]:
best_model = search.best_estimator_
predictions = best_model.predict(X_test)
metrics = {
    'model': MODEL_NAME,
    'features': FEATURES,
    'best_params': search.best_params_,
    'cv_macro_f1': float(search.best_score_),
    'test_accuracy': float(accuracy_score(y_test, predictions)),
    'test_balanced_accuracy': float(balanced_accuracy_score(y_test, predictions)),
    'test_macro_f1': float(f1_score(y_test, predictions, average='macro', zero_division=0)),
    'test_fatal_recall': float(recall_score(y_test, predictions, labels=[1], average=None, zero_division=0)[0]),
    'test_serious_recall': float(recall_score(y_test, predictions, labels=[2], average=None, zero_division=0)[0]),
    'test_slight_recall': float(recall_score(y_test, predictions, labels=[3], average=None, zero_division=0)[0]),
    'test_confusion_matrix': confusion_matrix(y_test, predictions, labels=[1, 2, 3]).tolist(),
}
(RESULTS / 'best_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame({'collision_index': test[GROUP], 'actual': y_test, 'predicted': predictions}).to_csv(RESULTS / 'test_predictions.csv', index=False)
joblib.dump(best_model, BEST / 'knn_best.pkl')
joblib.dump(best_model, MODEL_DIR / 'knn_2.pkl')
(BEST / 'knn_best_params.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))